In [ ]:
import numpy as np
import cv2
import os
from glob import glob
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50
from torchvision.models.segmentation.deeplabv3 import ASPP, DeepLabHead

In [ ]:
class RetinalDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=256, is_train=True):
        """
        适配DeepLab的视网膜数据集类
        :param image_dir: 图像目录路径
        :param mask_dir: 掩码目录路径
        :param img_size: 输入网络尺寸（保持与原代码一致）
        :param is_train: 训练模式标识
        """
        self.images, self.masks = self.load_dataset(image_dir, mask_dir)
        self.img_size = img_size
        self.retina_mean = [0.485, 0.456, 0.406]  # 视网膜专用归一化参数
        self.retina_std = [0.229, 0.224, 0.225]
    # 加载图像和mask
    def load_dataset(self,image_dir, mask_dir):
        images = []
        masks = []
        image_to_mask_mapping = {
            "1.jpg": "C0001273_EX.png",
            "2.jpg": "C0002369_EX.png",
            "3.jpg": "C0003164_EX.png",
            "4.jpg": "C0003166_EX.png",
            "5.jpg": "C0007155_EX.png",
            "6.jpg": "C0007158_EX.png",
            "7.jpg": "C0007248_EX.png",
            "8.jpg": "C0010059_EX.png",
            "9.jpg": "C0010362_EX.png",
            "10.jpg":"C0010452_EX.png",
            "11.jpg":"C0010492_EX.png",
            "12.jpg":"C0014759_EX.png",
            "13.jpg":"C0016193_EX.png",
            "14.jpg":"C0024407_EX.png",
            "15.jpg":"C0026111_EX.png"
        }
        for img_path in glob(os.path.join(image_dir, "*.*")):
            base_name = os.path.basename(img_path)
            if base_name.startswith("remove_illumination_"):
                mask_base_name = base_name.replace("remove_illumination_", "").replace(".bmp", "_EX.png")
            else:
                mask_base_name = image_to_mask_mapping.get(base_name, None)
            if mask_base_name is None:
                continue
            mask_path = os.path.join(mask_dir, mask_base_name)
            img = cv2.imread(img_path)
            mask = cv2.imread(mask_path, 0)
            if img is None or mask is None:
                continue
            # 统一分辨率并归一化
            img = cv2.resize(img, (256, 256))  # U-Net 输入尺寸
            mask = cv2.resize(mask, (256, 256))
            mask[mask > 0] = 1  # 二值化（渗出液=1，背景=0）
            images.append(img)
            masks.append(mask)
        return np.array(images), np.array(masks)
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        """适配DeepLab的数据格式要求"""
        # 转换颜色空间 BGR->RGB
        image = cv2.cvtColor(self.images[idx], cv2.COLOR_BGR2RGB)
        mask = self.masks[idx]

        # 归一化处理 (DeepLab需要浮点输入)
        image = image.astype(np.float32) / 255.0
        image = (image - self.retina_mean) / self.retina_std

        # 转换维度顺序为 [C, H, W]
        image = np.transpose(image, (2, 0, 1))
        mask = np.expand_dims(mask, axis=0)  # 添加通道维度

        return {
            'image': torch.from_numpy(image).float(),
            'mask': torch.from_numpy(mask).float()
        }

In [ ]:
# 数据路径
train_image_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/去光照/训练"
train_mask_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/groudtruth/训练"
test_image_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/去光照/测试"
test_mask_dir = "/mnt/e/平时作业/机器学习/大作业/眼底图像的渗出液分割/EX数据/groudtruth/测试"
# 创建数据集实例
train_dataset = RetinalDataset(train_image_dir, train_mask_dir)
test_dataset = RetinalDataset(test_image_dir, test_mask_dir)
 # 创建数据加载器
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
    
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=1,
    pin_memory=True
)
# 验证数据格式
sample = next(iter(train_loader))
print(f"图像张量尺寸: {sample['image'].shape}")  # 预期: [4, 3, 256, 256]
print(f"掩码张量尺寸: {sample['mask'].shape}")   # 预期: [4, 1, 256, 256]
print(f"像素值范围: {torch.min(sample['image'])} - {torch.max(sample['image'])}")

In [ ]:
class ImprovedDeepLabv3Plus(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        # Backbone初始化
        backbone = resnet50(pretrained=pretrained)
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = self._add_se(backbone.layer2)
        self.layer3 = self._add_se(backbone.layer3)
        self.layer4 = self._add_se(backbone.layer4)
        
        # 新增组件定义
        self.aspp = ASPP(2048, [3, 6, 9], 256)  # 需预先导入ASPP模块
        self.low_level_conv = nn.Sequential(
            nn.Conv2d(256, 48, 1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU()
        )
        self.decoder = DeepLabHead(256 + 48, num_classes)  # 需导入DeepLabHead

    def _add_se(self, layer):
        modules = list(layer.children())
        return nn.Sequential(
            *modules,
            SELayer(modules[-1].conv3.out_channels)
        )

    def forward(self, x):
        # 初始特征提取
        x = self.conv1(x)        # [b,3,256,256] → [b,64,128,128]
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)      # [b,64,64,64]
        
        # 获取中间特征
        x = self.layer1(x)       # [b,256,64,64]
        low_level_feat = x
        
        x = self.layer2(x)       # [b,512,32,32]
        x = self.layer3(x)       # [b,1024,16,16]
        x = self.layer4(x)       # [b,2048,8,8]
        
        # ASPP处理
        high_feat = self.aspp(x) # [b,256,8,8]
        
        # 调整上采样倍数（关键修正）
        high_feat = F.interpolate(
            high_feat, 
            size=low_level_feat.shape[2:],  # 直接指定目标尺寸为64x64
            mode='bilinear', 
            align_corners=True
        )
        
        # 处理低层特征
        low_feat = self.low_level_conv(low_level_feat)  # [b,48,64,64]
        
        # 特征融合
        combined = torch.cat([low_feat, high_feat], dim=1)  # [b, 48+256=304,64,64]
        
        # 解码输出
        output = self.decoder(combined)  # [b,1,64,64]
        
        # 最终上采样回原图尺寸
        return F.interpolate(
            output, 
            scale_factor=4,  # 64 * 4=256
            mode='bilinear', 
            align_corners=True
        )

In [ ]:
# SE注意力模块
class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

In [ ]:
# 组合损失函数（保持与UNet相同）
class CompositeLoss(nn.Module):
    def __init__(self, alpha=0.7):
        super().__init__()
        self.alpha = alpha  # Dice系数权重
        self.bce = nn.BCEWithLogitsLoss()
        
    def dice_loss(self, pred, target):
        smooth = 1e-5
        pred = torch.sigmoid(pred)
        intersection = (pred * target).sum(dim=(2,3))
        union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
        dice = (2. * intersection + smooth) / (union + smooth)
        return 1 - dice.mean()
    
    def forward(self, pred, target):
        return self.alpha * self.dice_loss(pred, target) + \
               (1 - self.alpha) * self.bce(pred, target)

In [ ]:
import time
import matplotlib.pyplot as plt
# 在模型定义之后添加设备检测
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# 初始化记录容器
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
epoch_times = []

# 模型初始化
model = ImprovedDeepLabv3Plus(num_classes=1).to(device)
criterion = CompositeLoss(alpha=0.7)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# 学习率调度器
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

def calculate_accuracy(pred, target):
    """计算像素级准确率"""
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()  # 二值化
    correct = (pred == target).sum().item()
    total = target.numel()  # 总像素数
    return correct / total

# 训练循环
for epoch in range(50):
    epoch_start = time.time()
    
    # 训练阶段
    model.train()
    total_train_loss = 0.0
    total_train_acc = 0.0
    for batch in train_loader:
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        # 累计指标
        total_train_loss += loss.item()
        total_train_acc += calculate_accuracy(outputs, masks)
    
    # 计算平均指标
    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_acc = total_train_acc / len(train_loader)
    
    # 验证阶段
    model.eval()
    total_val_loss = 0.0
    total_val_acc = 0.0
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            total_val_loss += loss.item()
            total_val_acc += calculate_accuracy(outputs, masks)
    
    avg_val_loss = total_val_loss / len(test_loader)
    avg_val_acc = total_val_acc / len(test_loader)
    
    # 记录指标
    train_losses.append(avg_train_loss)
    train_accuracies.append(avg_train_acc)
    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    
    # 更新学习率
    scheduler.step(avg_val_loss)
    
    # 打印信息
    print(f"Epoch {epoch+1:02d}/{50} | "
          f"Time: {epoch_time:.1f}s | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Train Acc: {avg_train_acc:.2%} | "
          f"Val Acc: {avg_val_acc:.2%}")

# 可视化训练过程
plt.figure(figsize=(12, 5))

# 准确率曲线
plt.subplot(1, 2, 1)
plt.plot(train_accuracies, label='Train Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# 损失曲线
plt.subplot(1, 2, 2)
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# 输出时间统计
print(f"\nTotal training time: {sum(epoch_times):.1f}s")
print(f"Average epoch time: {np.mean(epoch_times):.1f}s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc
from tqdm import tqdm

def evaluate_model(model, test_loader, device='cuda'):
    """综合评估函数"""
    model.eval()
    model.to(device)
    
    # 初始化存储容器
    all_preds = []
    all_labels = []
    dice_scores = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating'):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            
            # 存储原始数据（用于ROC）
            all_preds.append(probs.cpu().view(-1))
            all_labels.append(masks.cpu().view(-1))
            
            # 计算Dice系数
            preds = (probs > 0.5).float()
            intersection = (preds * masks).sum(dim=(1,2,3))
            union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
            dice = (2. * intersection) / (union + 1e-6)
            dice_scores.extend(dice.cpu().tolist())

    # 合并所有结果
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy().astype(int)
    
    return {
        'dice': dice_scores,
        'preds': all_preds,
        'labels': all_labels
    }

# 执行评估
results = evaluate_model(model, test_loader)

In [ ]:
### 1. DICE系数统计 ###
mean_dice = np.mean(results['dice'])
print(f"Mean Dice Score: {mean_dice:.4f}")

### 2. 混淆矩阵计算 ###
binary_preds = (results['preds'] > 0.5).astype(int)
cm = confusion_matrix(results['labels'], binary_preds)

plt.figure(figsize=(6,6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = [0, 1]
plt.xticks(tick_marks, ['Background', 'Exudate'])
plt.yticks(tick_marks, ['Background', 'Exudate'])
plt.xlabel('Predicted')
plt.ylabel('True')

# 添加数值标签
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")
plt.show()


In [ ]:
### 3. 召回率等指标计算 ###
tn, fp, fn, tp = cm.ravel()
recall = tp / (tp + fn)
precision = tp / (tp + fp)
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"Recall (Sensitivity): {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
### 4. ROC曲线绘制 ###
fpr, tpr, thresholds = roc_curve(results['labels'], results['preds'])
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()


In [ ]:

from sklearn.metrics import roc_auc_score
import numpy as np

def calculate_auc_details(y_true, y_pred, n_bootstrap=1000, ci=0.95):
    """
    增强版AUC计算（含置信区间）
    
    参数：
        y_true: 真实标签数组
        y_pred: 预测概率数组
        n_bootstrap: 自助采样次数
        ci: 置信水平
        
    返回：
        auc_dict: 包含AUC统计量的字典
    """
    # 基础AUC计算
    base_auc = roc_auc_score(y_true, y_pred)
    return {
        'auc': base_auc
    }

# 在您的评估代码部分调用
results = evaluate_model(model, test_loader)
auc_info = calculate_auc_details(results['labels'], results['preds'])

# 格式化输出结果
print(f"AUC = {auc_info['auc']:.3f} )")

In [ ]:
### 5. 可视化预测结果 ###
def plot_predictions(model, test_loader, num_samples=3):
    # 自动检测可用设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    
    with torch.no_grad():
        for idx, batch in enumerate(test_loader):
            if idx >= num_samples:
                break
                
            images = batch['image'].to(device)
            masks = batch['mask'].cpu().numpy()
            
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(float)
            
            # 原始图像
            axes[idx, 0].imshow(images[0].cpu().permute(1,2,0).numpy() * 0.5 + 0.5)
            axes[idx, 0].set_title('Original Image')
            
            # 真实标签
            axes[idx, 1].imshow(masks[0][0], cmap='gray')
            axes[idx, 1].set_title('Ground Truth')
            
            # 预测结果
            axes[idx, 2].imshow(preds[0][0], cmap='gray')
            axes[idx, 2].set_title('Prediction')
            
    plt.tight_layout()
    plt.show()

plot_predictions(model, test_loader)

In [ ]:
import joblib

# 保存模型到本地
joblib.dump(model, "deep_model.pkl")
print("Model saved as 'deep_model.pkl'")

In [ ]:
import pyvista as pv
import numpy as np
from matplotlib.cm import get_cmap

def plot_3d_segmentation(volume_3d, spacing=(1,1,3), opacity=0.3, color='red'):
    """修正后的三维可视化函数"""
    # 创建网格对象 (ImageData)
    grid = pv.ImageData()
    
    # 设置网格参数（关键修正点）
    grid.dimensions = np.array(volume_3d.shape)  # 维度应为数据形状（非形状+1）
    grid.spacing = spacing  # 物理间距
    grid.origin = (0, 0, 0)  
    
    # 将数据关联到顶点（Point Data）
    grid.point_data["labels"] = volume_3d.flatten(order="F")  # 注意Fortran顺序
    
    # 提取等值面
    contours = grid.contour([0.5], scalars="labels")  # 显式指定标量场
    
    # 可视化设置
    p = pv.Plotter()
    p.add_mesh(contours, 
               color=color, 
               opacity=opacity,
               smooth_shading=True)
    p.show()

In [ ]:

# DeepLabV3+预测结果处理
dl_pred_3d = (results['preds'] > 0.5).reshape(256, 256, -1)
plot_3d_segmentation(dl_pred_3d,
                        spacing=(0.1, 0.1, 0.5),
                        opacity=0.7,
                        color='#4169E1')  # 皇家蓝渲染